# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention

Notes:
* This file must be in the same folder as "utils.py"

In [1]:
# Housekeeping

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [2]:
# Dates
start_date = "11-01-2021"
end_date = "07-18-2025"
date_range = start_date + "--" + end_date

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
complete_files = originals + "complete/" + date_range + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it
metadata_folder = originals + "avian-influenza/metadata/"

## Read Metadata 

In [3]:
# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata

# print(len(metadata)) 
# display(metadata[metadata["geo_loc_name"] != "United States///"]) # ["geo_loc_name"])
# display(metadata)

10228
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')


In [4]:
# Get list of genotypes

# os.chdir(references)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

# genotypes = ["B3.13", "D1.1", "D1.3"]

genotypes = ["D1.1"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year, collection date, geo location, genotype

We need: host_type

host = Host

geo_loc_name = geo_loc_name

geo_location = country (abbreviated)-geo_loc_name (abbreviated) e.g. USA-MD

isolate = isolate

collection date = Collection_Date

serotype = H5N1 (hard-coded)

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

## Get genotype

In [5]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata = metadata[metadata["Genotype"].isin(genotypes)]

print(len(metadata)) 
# display(metadata)

2868


## Get specific geolocation

In [6]:
os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")

# forbidden_chars = [", ", ": "] # List of characters to replace
# Format: USA-[state abbreviation], e.g. USA-MD
metadata["Geo_Location"] = metadata["name_state"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    
                                                        lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                        else 
                                                        "USA"
                                                        )

# If USA-, delete -
metadata["Geo_Location"] = metadata["Geo_Location"].apply(lambda x: x.replace("-", "") if x == "USA-" else x)

# Rename variable back to metadata as we merge metadata and metadata_genbank
# metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
2135,SRR31597125,WGS,148.01,239713097,PRJNA1102327,SAMN45128756,Viral,85596786,USDA-NVSL,2024-11-05,...,Washington,2025-05-09_10-49-36,SRR31597125.fa,D1.1,"MP:ea3, HA:ea3, NA:am4N1, NS:ea3, PB1:ea3, PB2...","ea3:22-013001-001:MP, ea3:22-013001-001:HA, am...","100.00%, 99.41%, 99.73%, 99.28%, 99.52%, 99.96...","0, 10, 3, 6, 11, 1, 2, 1",Ran on FASTA - No Coverage Report,USA-WA
2136,SRR31597126,WGS,148.23,117600504,PRJNA1102327,SAMN45128755,Viral,42253221,USDA-NVSL,2024-11-05,...,Washington,2025-05-09_10-49-00,SRR31597126.fa,D1.1,"HA:ea3, NP:am13, NS:ea3, MP:ea3, NA:am4N1, PB2...","ea3:22-013001-001:HA, am13:24-030039-001:NP, e...","99.41%, 99.93%, 99.28%, 100.00%, 99.73%, 99.96...","10, 1, 6, 0, 3, 1, 11, 2",Ran on FASTA - No Coverage Report,USA-WA
2597,SRR31596955,WGS,147.84,30457528,PRJNA1102327,SAMN45128993,Viral,11103535,USDA-NVSL,2024,...,,2025-05-09_10-48-14,SRR31596955.fa,D1.1,"PB1:ea3, PA:am4, NP:am13, PB2:am24, MP:ea3, NS...","ea3:22-013001-001:PB1, am4:24-030039-001:PA, a...","99.47%, 99.83%, 99.80%, 99.78%, 99.80%, 99.17%...","12, 3, 3, 5, 2, 7, 4, 12",Ran on FASTA - No Coverage Report,USA
2598,SRR31596956,WGS,147.11,44241301,PRJNA1102327,SAMN45128992,Viral,16110232,USDA-NVSL,2024,...,,2025-05-09_10-49-27,SRR31596956.fa,D1.1,"NA:am4N1, MP:ea3, PB1:ea3, HA:ea3, PB2:am24, N...","am4N1:24-030039-001:NA, ea3:22-013001-001:MP, ...","99.62%, 99.80%, 99.47%, 99.30%, 99.78%, 99.80%...","4, 2, 12, 12, 5, 3, 3, 8",Ran on FASTA - No Coverage Report,USA
2599,SRR31596957,WGS,147.55,54588419,PRJNA1102327,SAMN45128991,Viral,19720626,USDA-NVSL,2024-10-30,...,California,2025-05-09_10-45-55,SRR31596957.fa,D1.1,"NS:ea3, MP:ea3, PB2:am24, NA:am4N1, PA:am4, HA...","ea3:22-013001-001:NS, ea3:22-013001-001:MP, am...","99.28%, 99.90%, 99.69%, 99.23%, 99.66%, 99.47%...","6, 1, 7, 8, 6, 9, 2, 14",Ran on FASTA - No Coverage Report,USA-CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10223,SRR34542495,WGS,145.74,109679874,PRJNA980729,SAMN49975075,Viral,37013777,USDA-NVSL,2025,...,,2025-07-19_06-19-09,SRR34542495.fa,D1.1,"HA:ea3, NA:am4N1, NS:ea3, PB1:ea3, NP:am13, PA...","ea3:22-013001-001:HA, am4N1:24-030039-001:NA, ...","99.30%, 99.04%, 99.17%, 99.33%, 99.67%, 99.49%...","12, 11, 7, 12, 5, 9, 5, 0",Ran on FASTA - No Coverage Report,USA
10224,SRR34542496,WGS,145.01,92718819,PRJNA980729,SAMN49975074,Viral,31474859,USDA-NVSL,2025,...,,2025-07-19_06-19-09,SRR34542496.fa,D1.1,"MP:ea3, NS:ea3, NA:am4N1, PB1:ea3, HA:ea3, PA:...","ea3:22-013001-001:MP, ea3:22-013001-001:NS, am...","100.00%, 99.05%, 99.23%, 99.06%, 99.24%, 99.49...","0, 8, 8, 17, 13, 9, 4, 5",Ran on FASTA - No Coverage Report,USA
10225,SRR34542497,WGS,147.70,92271701,PRJNA980729,SAMN49975073,Viral,31462934,USDA-NVSL,2025,...,,2025-07-19_06-19-09,SRR34542497.fa,D1.1,"NP:am13, HA:ea3, MP:ea3, NS:ea3, PA:am4, PB1:e...","am13:24-030039-001:NP, ea3:22-013001-001:HA, e...","99.53%, 99.24%, 100.00%, 99.05%, 99.49%, 99.34...","7, 13, 0, 8, 9, 12, 4, 14",Ran on FASTA - No Coverage Report,USA
10226,SRR34542498,WGS,146.93,125871082,PRJNA980729,SAMN49975072,Viral,43138274,USDA-NVSL,2025,...,,2025-07-19_06-19-09,SRR34542498.fa,D1.1,"PB2:am24, PA:am4, MP:ea3, NP:am13, PB1:ea3, NA...","am24:24-030039-001:PB2, am4:24-030039-001:PA, ...","99.74%, 99.49%, 99.90%, 99.53%, 99.39%, 99.13%...","6, 9, 1, 7, 11, 10, 8, 12",Ran on FASTA - No Coverage Report,USA


## Collection Dates

In [7]:
# Get years from collection dates
metadata["years"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x, default=datetime(2000, 1, 1), fuzzy=True).year if x == x else str(x)) # Get year only from collection date

print(metadata["Collection_Date"])

2135     2024-11-05
2136     2024-11-05
2597           2024
2598           2024
2599     2024-10-30
            ...    
10223          2025
10224          2025
10225          2025
10226          2025
10227          2025
Name: Collection_Date, Length: 2868, dtype: object


## Get host type

In [8]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x or "/" not in x or len(x.split("/")) < 2 # If NaN or split isolate doesn't exist or split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home + "references/")
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
           wild_avian domestic_avian               cattle        feline  \
0    great_horned_owl       pheasant            dairy_cow           cat   
1        common_raven         turkey               cattle  domestic_cat   
2       cooper's_hawk        chicken  cattle milk product     feral_cat   
3        coopers_hawk          goose          bovine_milk        feline   
4             peafowl    guinea_fowl              bovine   domestic-cat   
..                ...            ...                  ...           ...   
792       eared grebe            NaN                  NaN           NaN   
793          flamingo            NaN                  NaN           NaN   
794    swanson's_hawk            NaN                  NaN           NaN   
795       cape_petrel            NaN                  NaN           NaN   
796      king_penguin            NaN                  NaN           NaN   

      other_mammal       human         other  new  
0       deer mouse  washington         mixed

In [9]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

## Make names using all the attributes we collected

In [10]:
# Make names

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")[1]) > 0 else x.split("/")[0]) + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: x if "-" not in x else str(dateutil.parser.parse(x, default=datetime(2000, 1, 1)).strftime("%Y")) if dateutil.parser.parse(x, default=datetime(2000, 1, 1)).month == datetime(2000, 1, 1).month and dateutil.parser.parse(x, default=datetime(2000, 1, 1)).day == datetime(2000, 1, 1).day else dateutil.parser.parse(x, default=datetime(2000, 1, 1)).strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

2135     >SRR31597125|A/chicken/Washington/24-032809-00...
2136     >SRR31597126|A/chicken/Washington/24-032809-00...
2597     >SRR31596955|A/chicken/United States/24-032297...
2598     >SRR31596956|A/goose/United States/24-032297-0...
2599     >SRR31596957|A/chicken/California/24-032296-00...
                               ...                        
10223    >SRR34542495|A/mallard_duck/United States/25-0...
10224    >SRR34542496|A/mallard_duck/United States/25-0...
10225    >SRR34542497|A/chukar/United States/25-019408-...
10226    >SRR34542498|A/chukar/United States/25-019408-...
10227    >SRR34542499|A/chukar/United States/25-019408-...
Name: Name, Length: 2868, dtype: object

In [11]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="isolate", keep="first") # Isolates may be identical

In [12]:
print(metadata)
# metadata.to_csv("metadata_test.csv")

               Run Assay Type  AvgSpotLen      Bases    BioProject  \
2135   SRR31597125        WGS      148.01  239713097  PRJNA1102327   
2136   SRR31597126        WGS      148.23  117600504  PRJNA1102327   
2597   SRR31596955        WGS      147.84   30457528  PRJNA1102327   
2598   SRR31596956        WGS      147.11   44241301  PRJNA1102327   
2599   SRR31596957        WGS      147.55   54588419  PRJNA1102327   
...            ...        ...         ...        ...           ...   
10223  SRR34542495        WGS      145.74  109679874   PRJNA980729   
10224  SRR34542496        WGS      145.01   92718819   PRJNA980729   
10225  SRR34542497        WGS      147.70   92271701   PRJNA980729   
10226  SRR34542498        WGS      146.93  125871082   PRJNA980729   
10227  SRR34542499        WGS      148.06   96369330   PRJNA980729   

          BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
2135   SAMN45128756          Viral  85596786   USDA-NVSL      2024-11-05  ... 

## Make FASTA files

In [13]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [14]:
# Create fasta files 

os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty
        output_path = originals + "complete/" + pair + "_" + date_range + "_andersen.fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            name = name.replace(" ", "_")
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

>SRR31597125|A/chicken/Washington/24-032809-002/2024|H5N1|USA-WA|2024-11-05|domestic_avian|D1.1
>SRR31597126|A/chicken/Washington/24-032809-001/2024|H5N1|USA-WA|2024-11-05|domestic_avian|D1.1
>SRR31596955|A/chicken/United States/24-032297-002/2024|H5N1|USA|2024|domestic_avian|D1.1
>SRR31596956|A/goose/United States/24-032297-001/2024|H5N1|USA|2024|domestic_avian|D1.1
>SRR31596957|A/chicken/California/24-032296-002/2024|H5N1|USA-CA|2024-10-30|domestic_avian|D1.1
>SRR31596958|A/chicken/California/24-032296-001/2024|H5N1|USA-CA|2024-10-30|domestic_avian|D1.1
>SRR31596968|A/chicken/Washington/24-032192-001/2024|H5N1|USA-WA|2024-10-28|domestic_avian|D1.1
>SRR31596969|A/chicken/Washington/24-032191-004/2024|H5N1|USA-WA|2024-10-30|domestic_avian|D1.1
>SRR31596970|A/chicken/Washington/24-032191-003/2024|H5N1|USA-WA|2024-10-30|domestic_avian|D1.1
>SRR31596972|A/chicken/Washington/24-032191-002/2024|H5N1|USA-WA|2024-10-30|domestic_avian|D1.1
>SRR31596973|A/chicken/Washington/24-032191-001/2024|H